# 3 - MLTSA for dummies

This notebook is the shortest end-to-end introduction to the basic MLTSA loop:

1. get a dataset
2. decide which part of the trajectory to use
3. train a model
4. check validation and truly unseen test accuracy
5. inspect which features the model relied on

We keep one trajectory as one sample. That makes the train/validation/test split easy to
reason about and avoids leaking frames from the same trajectory across splits.

In [ ]:
from pathlib import Path
import sys

try:
    import mltsa  # noqa: F401
except ImportError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        src_dir = parent / "src"
        if (src_dir / "mltsa").exists():
            sys.path.insert(0, str(src_dir))
            break
    import mltsa  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

for parent in (Path.cwd(), *Path.cwd().parents):
    notebooks_dir = parent / "notebooks"
    if notebooks_dir.exists():
        DATA_DIR = notebooks_dir / "_generated"
        break
else:
    DATA_DIR = Path.cwd() / "_generated"

DATA_DIR.mkdir(exist_ok=True, parents=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

## Step 1: prepare the trajectory splits

We either reuse the 1D dataset from notebook 1 or create a small one on the fly. Then we
define an early-time window, split the available trajectories into train and validation,
and generate a separate set of unseen trajectories for the final test.

In [ ]:
from sklearn.model_selection import train_test_split

from mltsa.explain import analyze
from mltsa.models import get_model
from mltsa.synthetic import load_dataset, make_1d_dataset

dataset_path = DATA_DIR / "synthetic_1d_basics.h5"
if dataset_path.exists():
    dataset = load_dataset(dataset_path)
    print("Loaded existing dataset from notebook 1.")
else:
    dataset = make_1d_dataset(
        n_trajectories=144,  # Enough trajectories for train/validation
        n_steps=64,  # Total steps available in each trajectory
        n_features=12,  # Total observed features
        n_relevant=3,  # Hidden ground-truth relevant features
        base_seed=1234,  # Reproducible dataset definition
    )
    dataset.save(dataset_path, overwrite=True)
    print("Generated a fresh dataset because no local HDF5 was available.")

window = 24  # Early-time window used for the prediction problem
X_all = dataset.X[:, :window, :].reshape(dataset.n_trajectories, -1)
y_all = dataset.y
feature_names = tuple(
    f"{name}@t{step:03d}"
    for step in range(window)
    for name in dataset.feature_names
)

X_train, X_val, y_train, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.25,
    random_state=0,
    stratify=y_all,
)

test_dataset = dataset.generate_more(60)
X_test = test_dataset.X[:, :window, :].reshape(test_dataset.n_trajectories, -1)
y_test = test_dataset.y

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Unseen test shape:", X_test.shape)

The shapes above are the key check: train and validation come from the same dataset, while
the test set comes from brand-new trajectories generated later from the same system.

## Step 2: fit a simple model

A random forest is a nice starting point because it is fast, robust, and exposes native
feature importances.

In [ ]:
model = get_model(
    "random_forest",
    n_estimators=200,  # Forest size
    min_samples_leaf=2,  # Mild regularization
    random_state=0,  # Reproducible training
)
model.fit(X_train, y_train)

validation_accuracy = model.score(X_val, y_val)
test_accuracy = model.score(X_test, y_test)

print(f"Validation accuracy: {validation_accuracy:.3f}")
print(f"Unseen test accuracy: {test_accuracy:.3f}")

A test score in roughly the `0.8-0.9` range is a good sign here: the model is learning
real signal, but the problem is still non-trivial.

## Step 3: ask the model which features mattered

Native importance works for tree models without needing `X` or `y` again.

In [ ]:
explanation = analyze(
    model,
    method="native",
    feature_names=feature_names,
)

importance_by_feature = explanation.importances.reshape(window, dataset.n_features).mean(axis=0)
ranked_feature_ids = np.argsort(importance_by_feature)[::-1]

print("Ground-truth relevant features:", dataset.relevant_feature_names)
print(
    "Top recovered features:",
    [dataset.feature_names[index] for index in ranked_feature_ids[:5]],
)

In [ ]:
colors = [
    "#e76f51" if index in dataset.relevant_features else "#b8c0c8"
    for index in range(dataset.n_features)
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(dataset.n_features), importance_by_feature, color=colors)
ax.set_xticks(np.arange(dataset.n_features), dataset.feature_names, rotation=45, ha="right")
ax.set_ylabel("mean importance across time")
ax.set_title("Recovered feature importance")
plt.tight_layout()
plt.show()

The orange bars are the true relevant features. If the model and the explanation are doing
their job, those bars should stand out from the rest.